Die beiden TPI Dateien mit 300 m und 1500 m werden den Koordinaten der Sensoren nun zugewiesen.
Es ist zu beachten, dass das AOI der TPI Dateien bzw. des DOMs das kleinste ist und darauf basierend der RF-Algorithmus laufen wird.

In [1]:
import os
import pandas as pd
import rasterio
import numpy as np

# =========================================================
# 1. Pfade definieren
# =========================================================
master_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\IDW_hoch2\Exp2_Features_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"
tpi_300_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (DOM)\wuerzburg_tpi_300m.tif"
tpi_1500_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (DOM)\wuerzburg_tpi_1500m.tif"

# Ausgabedatei
output_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\IDW_hoch2\Exp2_Features_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"

# =========================================================
# 2. Master-Datensatz laden
# =========================================================
print("Lade Masterdatei...")
df_master = pd.read_csv(master_file, sep=";", decimal=",")

# Sicherstellen, dass UTM-Koordinaten als numerische Werte vorliegen
for col in ["x_utm", "y_utm"]:
    if df_master[col].dtype == object:
        df_master[col] = df_master[col].str.replace(",", ".", regex=False)
    df_master[col] = pd.to_numeric(df_master[col], errors="coerce")

# Neue Spalten initialisieren
df_master["tpi_300m"] = np.nan
df_master["tpi_1500m"] = np.nan

# =========================================================
# 3. Funktion zum sicheren Extrahieren von statischen Rastern
# =========================================================
def extract_static_raster(raster_path, col_name):
    if not os.path.exists(raster_path):
        print(f"WARNUNG: Datei nicht gefunden -> {raster_path}")
        return

    print(f"Extrahiere Daten aus {os.path.basename(raster_path)}...")

    # Filtere nur Zeilen, die gültige Koordinaten haben
    valid_mask = df_master["x_utm"].notna() & df_master["y_utm"].notna()

    # Erstelle eine Liste von (x, y) Koordinaten-Tupeln
    coords_list = list(zip(df_master.loc[valid_mask, "x_utm"], df_master.loc[valid_mask, "y_utm"]))

    if len(coords_list) == 0:
        return

    with rasterio.open(raster_path) as src:
        nodata_val = src.nodata
        sampled_values = []

        # Werte anstechen
        for val in src.sample(coords_list):
            v = val[0]
            # NoData und absurde Platzhalter rausfiltern
            if v == nodata_val or v > 1e9 or v < -1e9:
                sampled_values.append(np.nan)
            else:
                sampled_values.append(v)

        # Werte exakt den gültigen Zeilen zuweisen
        df_master.loc[valid_mask, col_name] = sampled_values

# =========================================================
# 4. TPI-Daten extrahieren
# =========================================================
extract_static_raster(tpi_300_path, "tpi_300m")
extract_static_raster(tpi_1500_path, "tpi_1500m")

# =========================================================
# 5. Speichern
# =========================================================
df_master.to_csv(output_file, index=False, sep=";", decimal=",")

print("-" * 60)
print(f"ERFOLG! Der Datensatz hat nun TPI-Features und liegt hier:\n{output_file}")
print("-" * 60)

Lade Masterdatei...
Extrahiere Daten aus wuerzburg_tpi_300m.tif...
Extrahiere Daten aus wuerzburg_tpi_1500m.tif...
------------------------------------------------------------
ERFOLG! Der Datensatz hat nun TPI-Features und liegt hier:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\IDW_hoch2\Exp2_Features_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv
------------------------------------------------------------
